# 01. Dataset Verification

Notebook này ghi lại kết quả của **Giai đoạn 1 — Dataset Verification**.

Vai trò của notebook:

- Không tải lại dataset từ HuggingFace.
- Không train model.
- Chỉ đọc các output đã sinh ra từ `scripts/verify_dataset.py`.
- Kiểm tra lại split, schema, label mapping và chất lượng dữ liệu.
- Ghi nhận kết luận để dùng cho báo cáo.

Pipeline giai đoạn trước:

```text
HuggingFace dataset
→ data/raw/
→ data/processed/*_standardized.csv
→ outputs/mappings/
→ outputs/tables/
→ outputs/reports/
```

## 1. Setup

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
MAPPINGS_DIR = ROOT / "outputs" / "mappings"
REPORTS_DIR = ROOT / "outputs" / "reports"
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

Project root: d:\project-ml-engineering\viedufeedback-robust


## 2. Kiểm tra các file đầu ra bắt buộc

In [2]:
required_files = {
    "raw_train": RAW_DIR / "train.csv",
    "raw_validation": RAW_DIR / "validation.csv",
    "raw_test": RAW_DIR / "test.csv",
    "raw_metadata": RAW_DIR / "dataset_source_metadata.json",
    "processed_train": PROCESSED_DIR / "train_standardized.csv",
    "processed_validation": PROCESSED_DIR / "validation_standardized.csv",
    "processed_test": PROCESSED_DIR / "test_standardized.csv",
    "processed_all": PROCESSED_DIR / "all_standardized.csv",
    "sentiment_mapping": MAPPINGS_DIR / "sentiment_label_mapping.json",
    "topic_mapping": MAPPINGS_DIR / "topic_label_mapping.json",
    "split_summary": TABLES_DIR / "split_summary.csv",
    "quality_summary": TABLES_DIR / "dataset_quality_summary.csv",
    "dataset_schema": REPORTS_DIR / "dataset_schema.json",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("All required Stage 1 files exist.")

,name,path,exists
0,raw_train,d:\project-ml-engineering\viedufeedback-robust...,True
1,raw_validation,d:\project-ml-engineering\viedufeedback-robust...,True
2,raw_test,d:\project-ml-engineering\viedufeedback-robust...,True
3,raw_metadata,d:\project-ml-engineering\viedufeedback-robust...,True
4,processed_train,d:\project-ml-engineering\viedufeedback-robust...,True
5,processed_validation,d:\project-ml-engineering\viedufeedback-robust...,True
6,processed_test,d:\project-ml-engineering\viedufeedback-robust...,True
7,processed_all,d:\project-ml-engineering\viedufeedback-robust...,True
8,sentiment_mapping,d:\project-ml-engineering\viedufeedback-robust...,True
9,topic_mapping,d:\project-ml-engineering\viedufeedback-robust...,True


All required Stage 1 files exist.


## 3. Split summary

Mục tiêu: xác nhận dataset có đủ `train`, `validation`, `test` và số mẫu từng split đúng với kết quả verification.

In [3]:
split_summary = pd.read_csv(TABLES_DIR / "split_summary.csv")
display(split_summary)

,split,num_rows,num_columns,duplicated_rows
0,train,11426,3,0
1,validation,1583,3,0
2,test,3166,3,0


## 4. Data quality summary

Mục tiêu: kiểm tra text rỗng, missing label và duplicate text sau khi chuẩn hóa.

In [4]:
quality_summary = pd.read_csv(TABLES_DIR / "dataset_quality_summary.csv")
display(quality_summary)

,split,num_rows,empty_text,missing_text,duplicated_text,missing_sentiment_label,missing_sentiment_name,missing_topic_label,missing_topic_name
0,train,11426,0,0,1,0,0,0,0
1,validation,1583,0,0,0,0,0,0,0
2,test,3166,0,0,0,0,0,0,0


## 5. Official label mappings

Mapping phải lấy từ HuggingFace `ClassLabel`, không tự suy đoán.

In [5]:
with open(MAPPINGS_DIR / "sentiment_label_mapping.json", "r", encoding="utf-8") as f:
    sentiment_mapping = json.load(f)

with open(MAPPINGS_DIR / "topic_label_mapping.json", "r", encoding="utf-8") as f:
    topic_mapping = json.load(f)

display(Markdown("### Sentiment mapping"))
display(sentiment_mapping)

display(Markdown("### Topic mapping"))
display(topic_mapping)

### Sentiment mapping

{'id2label': {'0': 'negative', '1': 'neutral', '2': 'positive'},
 'label2id': {'negative': 0, 'neutral': 1, 'positive': 2}}

### Topic mapping

{'id2label': {'0': 'lecturer',
  '1': 'training_program',
  '2': 'facility',
  '3': 'others'},
 'label2id': {'lecturer': 0,
  'training_program': 1,
  'facility': 2,
  'others': 3}}

## 6. Dataset schema

Mục tiêu: xác nhận cột gốc và cột chuẩn hóa dùng cho toàn bộ pipeline sau.

In [6]:
with open(REPORTS_DIR / "dataset_schema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

display(schema.get("official_schema", schema))

{'text_col': 'sentence',
 'sentiment_col': 'sentiment',
 'topic_col': 'topic',
 'raw_columns': ['sentence', 'sentiment', 'topic'],
 'standardized_columns': ['id',
  'split',
  'text',
  'sentiment_label',
  'sentiment_name',
  'topic_label',
  'topic_name']}

## 7. Kiểm tra dữ liệu chuẩn hóa

Các giai đoạn sau sẽ đọc `data/processed/*_standardized.csv`, không đọc trực tiếp HuggingFace.

In [7]:
train_df = pd.read_csv(PROCESSED_DIR / "train_standardized.csv")
valid_df = pd.read_csv(PROCESSED_DIR / "validation_standardized.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_standardized.csv")

display(train_df.head())
print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

expected_cols = [
    "id",
    "split",
    "text",
    "sentiment_label",
    "sentiment_name",
    "topic_label",
    "topic_name",
]

missing_cols = [col for col in expected_cols if col not in train_df.columns]
if missing_cols:
    raise ValueError(f"Missing standardized columns: {missing_cols}")

print("Standardized schema is valid.")

,id,split,text,sentiment_label,sentiment_name,topic_label,topic_name
0,train_0,train,slide giáo trình đầy đủ .,2,positive,1,training_program
1,train_1,train,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,positive,0,lecturer
2,train_2,train,đi học đầy đủ full điểm chuyên cần .,0,negative,1,training_program
3,train_3,train,chưa áp dụng công nghệ thông tin và các thiết ...,0,negative,0,lecturer
4,train_4,train,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",2,positive,0,lecturer


Train shape: (11426, 7)
Validation shape: (1583, 7)
Test shape: (3166, 7)
Standardized schema is valid.


## 8. Kết luận Stage 1

Giai đoạn Dataset Verification hoàn thành khi các điểm sau đúng:

```text
- Dataset có đủ 3 split: train, validation, test.
- Cột text gốc là sentence và được chuẩn hóa thành text.
- Sentiment có 3 lớp: negative, neutral, positive.
- Topic có 4 lớp: lecturer, training_program, facility, others.
- Mapping label được lấy từ ClassLabel chính thức.
- Raw snapshot được lưu trong data/raw/.
- Dữ liệu chuẩn hóa được lưu trong data/processed/.
- Các giai đoạn sau dùng schema chuẩn hóa, không dùng trực tiếp schema gốc.
```

Ý nghĩa:

```text
Stage 1 tạo data contract cho toàn bộ dự án.
Nếu không có bước này, các kết quả baseline, PhoBERT và robustness evaluation có thể sai do nhầm split, nhầm label hoặc nhầm schema.
```